In [ ]:
import pandas as pd
import re

# Processes annotation and sample data, organizes it by extract_id, and applies a majority vote to determine the final score for each sample. 
# It also processes the text column to replace double asterisks, convert bold italic words to normal text, and remove inner asterisks.
def process_data(annotations_path, samples_path, output_path):
    # Load annotations and samples
    annotations = pd.read_csv(annotations_path)
    samples = pd.read_csv(samples_path)

    # Organize annotations by extract_id
    annotations_dict = {}
    for index, row in annotations.iterrows():
        extract_id = row["extract_id"]
        if extract_id not in annotations_dict:
            annotations_dict[extract_id] = []
        annotations_dict[extract_id].append(row["score"])

    # Organize samples by extract_id and filter by annotations_dict
    grouped_samples = samples.groupby('extract_id').agg({'target': 'first', 'target_compound': 'first', 'text': 'first'})
    ready_samples = grouped_samples[grouped_samples.index.isin(annotations_dict.keys())]

    # Add annotations and majority vote score
    ready_samples["annotations"] = ready_samples.index.map(lambda x: annotations_dict[x])
    ready_samples["mv_score"] = ready_samples["annotations"].apply(lambda x: majority_vote(sum(x) / len(x)))

    # Process text column
    ready_samples['text'] = ready_samples['text'].apply(process_text)

    # Save the processed data
    ready_samples.reset_index().to_csv(output_path, index=False)

# Get the majority vote based on the proportion of positive scores.
def majority_vote(proportion: float) -> float:
    """Get the majority vote."""
    if proportion == 0.5:
        return 0.5
    elif proportion > 0.5:
        return 1
    else:
        return 0

# Process text by replacing double asterisks, converting bold italic words to normal text, and removing inner asterisks.
def process_text(text):
    """Process text by replacing double asterisks, converting bold italic words, and removing inner asterisks."""
    bold_italic_to_normal = str.maketrans(
        "𝙖𝙗𝙘𝙙𝙚𝙛𝙜𝙝𝙞𝙟𝙠𝙡𝙢𝙣𝙤𝙥𝙦𝙧𝙨𝙩𝙪𝙫𝙬𝙭𝙮𝙯𝘼𝘽𝘾𝘿𝙀𝙁𝙂𝙃𝙄𝙅𝙆𝙇𝙈𝙉𝙊𝙋𝙌𝙍𝙎𝙏𝙐𝙑𝙒𝙓𝙔𝙕",
        "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    )
    text = text.replace("**", "*")  # Replace double asterisks
    text = re.sub(r"[𝙖-𝙯𝘼-𝙕]+", lambda match: f"**{match.group().translate(bold_italic_to_normal)}**", text)  # Convert bold italic words
    while re.search(r'\*\*(.*?)\*\*(.*?)\*\*', text):  # Remove inner asterisks
        text = re.sub(r'\*\*(.*?)\*\*(.*?)\*\*', r'**\1\2**', text)
    return text

In [ ]:
# Generate dataset of all 2090 samples

annotations_path = "./ready_annotations.csv"
samples_path = "./data.csv"
output_path = "./2090_ready_samples.csv"
process_data(annotations_path, samples_path, output_path)

In [ ]:
# Generate dataset of 1254 consensus samples

annotations_path = "./ready_annotations_1254.csv"
samples_path = "./data.csv"
output_path = "./1254_ready_samples.csv"
process_data(annotations_path, samples_path, output_path)